# RoboDream demo

This notebook downloads the released RoboDream weights and reproduces the three result types from the [project page](https://junjieye.com/RoboDream/) on the inputs shipped in [`demos/`](../demos):

1. **Retrieval and rebirth**: DROID trajectories retrieved for a new task are re-rendered from our lab camera and *reborn* in our scene with new objects.
2. **Prop-free teleoperation**: a trajectory teleoperated without any objects is turned into demonstrations with different objects.
3. **Compositional generation**: the same trajectory with a different object instance, scene, task or camera view.

Each demo case contains the model inputs that the full pipeline (`generation/`) produces: the robot-only renders of the external and wrist camera (Isaac Lab replay), the end-effector trajectory, the scene prior, the object prior and the instruction. Run the notebook inside the RoboDream container (`bash shell/run_docker.sh`) on one GPU (peak memory about 7 GB). Generation takes about 50 seconds per 189-frame clip on an A100, about 10 minutes for all cases; finished cases are saved and reused.

In [ ]:
import os, sys
ROOT = os.path.abspath('..')
sys.path.insert(0, ROOT)

import torch
from IPython.display import HTML, Video, display
from huggingface_hub import hf_hub_download

from robodream import demo as D
from robodream.generate import RoboDreamGenerator

CKPT_DIR = os.environ.get('ROBODREAM_CKPT_DIR', os.path.join(ROOT, 'checkpoints'))
OUT_DIR = os.path.join(os.environ.get('ROBODREAM_OUTPUT_DIR', os.path.join(ROOT, 'outputs')), 'demos')
DEMO_DIR = os.path.join(ROOT, 'demos')
os.makedirs(OUT_DIR, exist_ok=True)

def show(path, width=960):
    display(Video(path, embed=True, width=width, html_attributes='controls autoplay loop muted'))

## 1. Download the weights

* `Jay-Ye/RoboDream`: the RoboDream DiT (`model.pt`) and the default instruction embedding.
* `nvidia/Cosmos-Predict2-2B-Video2World`: only the video tokenizer (`tokenizer/tokenizer.pth`). Accept its license on Hugging Face and run `huggingface-cli login` first if the download is refused.

The instructions of the demo cases are embedded with the cached T5-11B embeddings in `demos/t5_prompts.pkl.gz`, so T5-11B is not needed here.

In [ ]:
model_path = hf_hub_download('Jay-Ye/RoboDream', 'model.pt', local_dir=os.path.join(CKPT_DIR, 'robodream'))
default_text_path = hf_hub_download('Jay-Ye/RoboDream', 'default_text_emb.pt', local_dir=os.path.join(CKPT_DIR, 'robodream'))
tokenizer_path = hf_hub_download('nvidia/Cosmos-Predict2-2B-Video2World', 'tokenizer/tokenizer.pth',
                                 local_dir=os.path.join(CKPT_DIR, 'nvidia', 'Cosmos-Predict2-2B-Video2World'))

generator = RoboDreamGenerator(
    checkpoint_path=model_path,
    tokenizer_path=tokenizer_path,
    t5_cache_path=os.path.join(DEMO_DIR, 't5_prompts.pkl.gz'),
    default_text_path=default_text_path,
    device='cuda',
)
manifest = D.load_manifest(DEMO_DIR)
cases = {name: D.load_case(DEMO_DIR, manifest, name) for name in manifest['cases']}
for (name, c) in cases.items():
    print(f'{name:45s} {c["mode"]:18s} {len(c["trajectory"]):4d} frames  {c["prompt"]!r}')

## 2. What the model sees

The inputs of one case: robot-only renders of the external and wrist camera, the scene prior (the workspace with the task objects removed) and the object prior (the objects to place in it). The end-effector trajectory (xyz, rpy, gripper per frame) is passed alongside.

In [ ]:
c = cases['retrieval_put_the_marker_in_the_bowl']
T = len(c['trajectory'])
inputs = D.columns([
    (D.two_views(c['render_external'], c['render_wrist'], T), 'Robot-only render', 'white'),
    (D.stack([c['scene_prior'], c['object_prior']], T, axis=1), 'Scene / object prior', 'white'),
], T)
D.write_video(os.path.join(OUT_DIR, 'inputs_example.mp4'), inputs)
show(os.path.join(OUT_DIR, 'inputs_example.mp4'), width=640)

In [ ]:
def generate(name, overwrite=False):
    '''Run one case (or reuse its saved videos) and return {'external', 'wrist'} uint8 videos.'''
    result = None if overwrite else D.load_result(OUT_DIR, name)
    if result is None:
        result = D.run_case(generator, cases[name], seed=0)
        D.save_result(OUT_DIR, name, result)
    return result

results = {}

## 3. Retrieval and rebirth

For each task, a DROID episode retrieved by instruction similarity was replayed in Isaac Lab from our calibrated lab camera. RoboDream regenerates it in our scene with our objects. *Retrieved* shows the original DROID episode (wrist camera on top).

In [ ]:
retrieval = [c for c in cases.values() if c['mode'] == 'retrieval_rebirth']
for c in retrieval:
    results[c['name']] = generate(c['name'])
references = {c['name']: D.read_video(os.path.join(DEMO_DIR, manifest['references'][c['name'].replace('retrieval_', 'retrieved_')]['path']))
              for c in retrieval}
fig = D.retrieval_figure(retrieval, results, references)
D.write_video(os.path.join(OUT_DIR, 'figure_retrieval_rebirth.mp4'), fig)
show(os.path.join(OUT_DIR, 'figure_retrieval_rebirth.mp4'), width=720)

## 4. Prop-free teleoperation

The operator performed the pick-and-place motion over an empty table. The same trajectory becomes three demonstrations by changing only the object prior (and the instruction).

In [ ]:
prop_free = [cases[n] for n in ['propfree_marker_bowl', 'propfree_novel_instance', 'propfree_novel_task']]
for c in prop_free:
    results[c['name']] = generate(c['name'])
teleop = D.read_video(os.path.join(DEMO_DIR, manifest['references']['propfree_teleop']['path']))
fig = D.prop_free_figure(teleop, prop_free, results)
D.write_video(os.path.join(OUT_DIR, 'figure_prop_free.mp4'), fig)
show(os.path.join(OUT_DIR, 'figure_prop_free.mp4'))

## 5. Compositional generation

One trajectory, interpreted in several ways by swapping a single input: the object prior (novel instance), the scene prior (novel scene), the object prior and instruction (novel task), or the camera (novel view: the trajectory is re-rendered from another camera and paired with a scene prior from that view).

In [ ]:
for n in ['propfree_novel_scene', 'propfree_novel_view']:
    results[n] = generate(n)
variants = [(cases['propfree_marker_bowl'], 'Generated demo', 'gray', 'object_prior'),
            (cases['propfree_novel_instance'], 'w/ Novel instance', 'blue', 'object_prior'),
            (cases['propfree_novel_scene'], 'w/ Novel scene', 'green', 'scene_prior'),
            (cases['propfree_novel_task'], 'w/ Novel task', 'red', 'object_prior'),
            (cases['propfree_novel_view'], 'w/ Novel view', 'orange', 'scene_prior')]
fig = D.compositional_figure(cases['propfree_marker_bowl'], variants, results)
D.write_video(os.path.join(OUT_DIR, 'figure_compositional.mp4'), fig)
show(os.path.join(OUT_DIR, 'figure_compositional.mp4'))

## 6. Your own inputs

To mix your own priors with a shipped trajectory, replace `scene_prior` / `object_prior` with any (H, W, 3) image of the external camera view. The object prior is a black canvas with the objects pasted in (`generation/tools/extract_priors.py` builds both priors from a photo of your scene). Instructions outside `demos/t5_prompts.pkl.gz` fall back to the default embedding unless you pass `t5_path` (T5-11B) to `RoboDreamGenerator`.

In [ ]:
c = dict(cases['propfree_marker_bowl'])
c['scene_prior'] = cases['propfree_novel_scene']['scene_prior']
c['object_prior'] = cases['propfree_novel_task']['object_prior']
c['prompt'] = cases['propfree_novel_task']['prompt']
custom = D.run_case(generator, c, seed=1)
D.write_video(os.path.join(OUT_DIR, 'custom.mp4'), D.two_views(custom['external'], custom['wrist'], len(c['trajectory'])))
show(os.path.join(OUT_DIR, 'custom.mp4'), width=320)